# 02 — Build ORACLE Extractive Labels

Greedy ORACLE: for each paper, pick up to 7 sentences whose concatenation maximizes ROUGE-1+ROUGE-2 vs the abstract. These become training labels for BERTSUM.

In [ ]:
# Setup: clone repo + mount Drive for persistent data
REPO_URL = 'https://github.com/Captain-Uchiha/scientific-paper-summarizer.git'
!rm -rf /content/scientific-summarizer
!git clone -q {REPO_URL} /content/scientific-summarizer
import sys, os
sys.path.insert(0, '/content/scientific-summarizer')
os.chdir('/content/scientific-summarizer')

!pip -q install rouge-score tqdm

from google.colab import drive
drive.mount('/content/drive', force_remount=True)
PROJECT_DIR = '/content/drive/MyDrive/scientific-summarizer-data'

In [ ]:
import json, os
from tqdm import tqdm
from preprocessing.oracle import greedy_oracle

IN_DIR  = f'{PROJECT_DIR}/dataset/processed'
OUT_DIR = f'{PROJECT_DIR}/dataset/processed'

def add_oracle(in_path, out_path, max_sents=7):
    with open(in_path) as f, open(out_path, 'w', encoding='utf-8') as out:
        for line in tqdm(f, desc=os.path.basename(in_path)):
            r = json.loads(line)
            selected, labels = greedy_oracle(r['sentences'], r['target_text'], max_sents=max_sents)
            r['ext_labels'] = labels
            r['ext_selected'] = selected
            out.write(json.dumps(r) + '\n')

add_oracle(f'{IN_DIR}/train.jsonl', f'{OUT_DIR}/train.oracle.jsonl')
add_oracle(f'{IN_DIR}/val.jsonl',   f'{OUT_DIR}/val.oracle.jsonl')
add_oracle(f'{IN_DIR}/test.jsonl',  f'{OUT_DIR}/test.oracle.jsonl')
print('Done.')